# MP5: Training Your Diffusion Model!

## Setup environment

In [1]:
# Import essential modules. Feel free to add whatever you need.
import matplotlib.pyplot as plt
import torch
from torch import optim 
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST, ImageFolder
from torchvision.transforms import ToTensor
from torchvision import transforms
from tqdm.notebook import tqdm # For progress bars
import os
import random

In [2]:
# CELL: Constants
# MODIFY: Constants for ImageNette
IMAGE_SIZE = 64 # Choose target image size (e.g., 64, 128)
IN_CHANNELS = 3 # ImageNette has RGB images
NUM_CLASSES = 10 # ImageNette has 10 classes
DATASET_PATH = './imagenette2-160' # CHANGE THIS to your ImageNette path

# ADD: ImageNet normalization constants (commonly used)
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]


## Visualization helper function

In [3]:
def denormalize(tensor):
    tensor = tensor.clone()
    # Use view to ensure broadcasting works correctly for (C,) constants with (N, C, H, W) tensor
    mean = torch.tensor(NORM_MEAN, device=tensor.device).view(1, -1, 1, 1)
    std = torch.tensor(NORM_STD, device=tensor.device).view(1, -1, 1, 1)
    tensor.mul_(std).add_(mean)
    return tensor

def visualize_images_with_titles(images: torch.Tensor, column_names: list[str]):
    """
    Visualize images as a grid and title the columns with the provided names.

    Args:
        images: (N, C, H, W) tensor of images, where N is (number of rows * number of columns)
        column_names: List of column names for the titles.

    Example usage:
    visualize_images_with_titles(torch.randn(16, 3, 64, 64), ['1', '2', '3', '4']) # Example for 3 channels
    """
    num_images, num_columns = images.shape[0], len(column_names)
    assert num_images % num_columns == 0, 'Number of images must be a multiple of the number of columns.'
    assert images.shape[1] == IN_CHANNELS, f'Input tensor should have {IN_CHANNELS} channels.'

    num_rows = num_images // num_columns
    fig, axes = plt.subplots(num_rows, num_columns, figsize=(num_columns * 2.5, num_rows * 2.5)) # Adjusted size

    if num_rows == 1 and num_columns == 1:
        axes = [[axes]] # Make it 2D iterable
    elif num_rows == 1:
        axes = [axes] # Make it list of axes
    elif num_columns == 1:
         axes = [[ax] for ax in axes] # Make it list of lists

    axes_flat = [ax for row in axes for ax in row] # Flatten axes for easy iteration

    for i, ax in enumerate(axes_flat):
        img_to_show = images[i]
        # Denormalize before showing
        img_to_show = denormalize(img_to_show)
        # Clamp values to [0, 1] after denormalization
        img_to_show = torch.clamp(img_to_show, 0, 1)
        # Permute C, H, W -> H, W, C for matplotlib
        img = img_to_show.permute(1, 2, 0).cpu().detach().numpy()

        ax.imshow(img)
        ax.axis('off')
        # Set title only for the first row
        if i < num_columns:
            ax.set_title(column_names[i % num_columns])

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()



# Part 1: Training a Single-step Denoising UNet


## Implementing Simple and Composed Ops

In [ ]:

class Conv(nn.Module):
    """
    Conv is a convolutional layer that doesn't change the image resolution, only the channel dimension.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Conv2d(3,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        """
        conv_result = self.conv(x)
        bn_result = self.bn(conv_result)
        gelu_result = self.gelu(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2], x.shape[3]), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2], x.shape[3])}, but got {gelu_result.shape}"
        return gelu_result


class DownConv(nn.Module):
    """
    DownConv is a convolutional layer that downsamples the tensor by 2.
    D_in, H, W => D_out, H/2, W/2
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=2, padding=1)
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Conv2d (3,2,1) + BN + GELU"""
        # Conv2d(3,2,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = self.conv(x)
        bn_result = self.bn(conv_result)
        gelu_result =self.gelu(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2)}, but got {gelu_result.shape}"
        return gelu_result


class UpConv(nn.Module):
    """
    UpConv is a convolutional layer that upsamples the tensor by 2.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.conv_transpose = nn.ConvTranspose2d(in_channels=in_channels, out_channels=out_channels, kernel_size=4, stride=2, padding=1)
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ConTranspose2d (4,2,1) + BN + GELU"""
        # ConvTranspose2d(4,2,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = self.conv_transpose(x)
        bn_result = self.bn(conv_result)
        gelu_result = self.gelu(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2)}, but got {gelu_result.shape}"
        return gelu_result


class Flatten(nn.Module):
    """
    Flatten is an average pooling layer that flattens a 7x7 tensor into a 1x1 tensor.
    7 is the resulting height and width after the downsampling operations.
    """

    def __init__(self):
        super().__init__()
        self.avg_pool = nn.AvgPool2d(kernel_size=7)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """AvgPool (7) + GELU"""
        # AvgPool(7) + GELU (nn.GELU())
        avg_pool_result = self.avg_pool(x)
        gelu_result = self.gelu(avg_pool_result)
        # assert gelu_result.shape[-2:] == (x.shape[0], x.shape[1]), \
        #     f"Expected shape {x.shape[0], x.shape[1]}, but got {gelu_result.shape[0], gelu_result.shape[1]}"
        return gelu_result


class Unflatten(nn.Module):
    """
    Unflatten is a convolutional layer that unflattens/upsamples a 1x1 tensor into a 7x7 tensor.
    """
    def __init__(self, in_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.conv_transpose = nn.ConvTranspose2d(in_channels=in_channels, out_channels=in_channels, kernel_size=7, stride=1, padding=0)
        self.bn = nn.BatchNorm2d(num_features=in_channels)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ConvTranspose2d(7,7,0) + BN + GELU"""
        # ConvTranspose2d(7,7,0) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = self.conv_transpose(x)
        bn_result = self.bn(conv_result)
        gelu_result =self.gelu(bn_result)

        assert gelu_result.shape[-2:] == (x.shape[0], self.in_channels), \
            f"Expected shape {(x.shape[0], self.in_channels)}, but got {gelu_result.shape[-2:]}"
        return gelu_result


class ConvBlock(nn.Module):
    """
    similar to Conv but includes an additional Conv. 
    Note that it has the same input and output shape as (1) Conv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(num_features=out_channels)
        self.gelu1 = nn.GELU()
        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(num_features=out_channels)
        self.gelu2 = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Conv2d(3,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU()) + Conv2d(1,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        """
        conv_result = self.conv1(x)
        bn_result = self.bn1(conv_result)
        gelu_result = self.gelu1(bn_result)
        conv_result_2 = self.conv2(gelu_result)
        bn_result_2 = self.bn2(conv_result_2)
        gelu_result_2 = self.gelu2(bn_result_2)

        # Check if the output shape is as expected
        assert gelu_result_2.shape == (x.shape[0], self.out_channels, x.shape[2], x.shape[3]), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2], x.shape[3])}, but got {gelu_result_2.shape}"
        return gelu_result_2


class DownBlock(nn.Module):
    """
    similar to DownConv but includes an additional ConvBlock. 
    Note that it has the same input and output shape as (2) DownConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        self.down_conv = DownConv(in_channels, out_channels)
        self.conv_block = ConvBlock(out_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """DownConv + ConvBlock"""
        # DownConv + ConvBlock
        downconv_result = self.down_conv(x)
        convblock_result = self.conv_block(downconv_result)

        # Check if the output shape is as expected      
        assert convblock_result.shape == (x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2)}, but got {convblock_result.shape}"
        
        return convblock_result


class UpBlock(nn.Module):
    """
    similar to UpConv but includes an additional ConvBlock. 
    Note that it has the same input and output shape as (3) UpConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        self.up_conv = UpConv(in_channels, out_channels)
        self.conv_block = ConvBlock(out_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """UpConv + ConvBlock"""
        # UpConv + ConvBlock
        upconv_result = self.up_conv(x)
        convblock_result = self.conv_block(upconv_result)

        # Check if the output shape is as expected
        assert convblock_result.shape == (x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2)}, but got {convblock_result.shape}"
        return convblock_result

## Implementing Unconditional UNet

In [5]:
class UnconditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int = IN_CHANNELS,
        num_hiddens: int = 32,
    ):
        self.in_channels = in_channels
        self.num_hiddens = num_hiddens

        super().__init__()

        # Layers in U-Net
        self.first_conv = ConvBlock(self.in_channels, num_hiddens)

        # Downsampling Path
        self.down1 = DownBlock(num_hiddens, num_hiddens)
        self.down2 = DownBlock(num_hiddens, num_hiddens * 2)

        # Flatten and Unflatten (bottom of U-Net)
        self.flatten = Flatten()
        self.unflatten = Unflatten(num_hiddens * 2)

        # Upsampling Path
        # upsample1 is the one that receives the concatenatino of Unflatten and DownBlock
        self.upsample1 = UpBlock(num_hiddens * 4, num_hiddens)
        self.upsample2 = UpBlock(num_hiddens * 2, num_hiddens)

        # Final Convolution Blocks
        self.final_conv1 = ConvBlock(num_hiddens * 2, num_hiddens)
        self.final_conv2 = nn.Conv2d(num_hiddens, self.in_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-2:] == (IMAGE_SIZE, IMAGE_SIZE),\
             f"Expect input shape to be ({IMAGE_SIZE}, {IMAGE_SIZE}). Got {x.shape[-2:]}"

        # Encoder?
        c1 = self.first_conv(x)
        d1 = self.down1(c1)
        d2 = self.down2(d1)

        # Bottleneck
        flat = self.flatten(d2)
        b = self.unflatten(flat)

        # Decoder
        cat1 = torch.cat((b, d2), dim=1)
        u1 = self.upsample1(cat1)

        ## Second level
        cat2 = torch.cat((u1, d1), dim=1)
        u2 = self.upsample2(cat2)

        ## Final (top) level
        cat3 = torch.cat((u2, c1), dim=1)
        f1 = self.final_conv1(cat3)

        # Final convolution
        out_res = self.final_conv2(f1)

        assert out_res.shape == x.shape, f"Output shape {out_res.shape} doesn't match input shape {x.shape}"

        return out_res

## Visualizing the noising process

In [6]:
# dataset = MNIST(root="data", download=True, transform=ToTensor(), train=True)

# train_loader = DataLoader(dataset, batch_size=16, shuffle=True)
# # Get a batch of data
# for images, labels in train_loader:
#     break
# # Create an instance of the model
# model = UnconditionalUNet(in_channels=1, num_hiddens=16)



# # Forward pass
# output = model(images)
# # Visualize the output
# visualize_images_with_titles(output, ['Output 1', 'Output 2', 'Output 3', 'Output 4'])
# # Visualize the input
# visualize_images_with_titles(images, ['Input 1', 'Input 2', 'Input 3', 'Input 4'])
# # Visualize the labels by printing them
# print("Labels (4x4 matrix):")
# print(labels.reshape(4, 4))


In [7]:
# ITERATIONS = 9
# NUM_IMAGES = 5

# dataset = MNIST(root="data", download=True, transform=ToTensor(), train=True)
# train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# for images, labels in train_loader:
#     break
# # Create an instance of the model
# model = UnconditionalUNet(in_channels=1, num_hiddens=16)

# with torch.no_grad():
#     # Select the first {NUM_IMAGES} images in the batch
#     imgs = images[:NUM_IMAGES]
#     progression = [imgs.clone()]

#     # Pass the images through the model for {ITERATIONS} iterations
#     for i in range(ITERATIONS):
#         imgs = model(imgs)
#         progression.append(imgs.clone())

# # Plot the original images and their progression
# num_iters = len(progression)  # 9 iterations (initial + ITERATIONS steps)
# num_imgs = progression[0].shape[0]  # NUM_IMAGES images

# fig, axes = plt.subplots(num_imgs, num_iters, figsize=(num_iters * 2, num_imgs * 2))
# for i in range(num_imgs):
#     for j in range(num_iters):
#         axes[i, j].imshow(progression[j][i].squeeze().cpu(), cmap="gray")
#         axes[i, j].axis("off")
#         if i == 0:
#             axes[i, j].set_title(f"Iter {j}")
# plt.tight_layout()
# plt.show()

## Training a Single-Step Unconditional UNet

- Plot the loss curve
- Sample results on the test set

In [8]:
# Define transforms using ImageNette constants
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE), # Resize smaller edge to IMAGE_SIZE
    transforms.CenterCrop(IMAGE_SIZE), # Crop center to IMAGE_SIZE x IMAGE_SIZE
    transforms.RandomHorizontalFlip(), # Add data augmentation
    transforms.ToTensor(), # Convert to tensor (0-1 range)
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD) # Normalize
])

test_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])


EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_HIDDENS = 64
NOISE_SIGMA = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: mps


In [9]:

# DataLoaders
train_dataset = ImageFolder(
    root=os.path.join(DATASET_PATH, 'train'),
    transform=train_transforms
)
test_dataset = ImageFolder(
    root=os.path.join(DATASET_PATH, 'val'),
    transform=test_transforms
)


train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)

# Visualization subset from the validation set
num_vis_images = 8
vis_subset_indices = random.sample(range(len(test_dataset)), num_vis_images)
test_subset = Subset(test_dataset, vis_subset_indices)
# Load all vis images at once, ensure batch size matches num_vis_images
vis_dataloader = DataLoader(test_subset, batch_size=num_vis_images)

# --- Instantiate Model ---
# MODIFY: Ensure correct in_channels is used
model = UnconditionalUNet(in_channels=IN_CHANNELS, num_hiddens=NUM_HIDDENS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

# --- Visualization Setup ---
# Get a fixed batch for visualization
fixed_vis_images, _ = next(iter(vis_dataloader))
fixed_vis_images = fixed_vis_images.to(DEVICE)
# Generate fixed noise ONCE for consistent visualization comparison
fixed_noise = torch.randn_like(fixed_vis_images) * NOISE_SIGMA
fixed_noisy_vis_images = fixed_vis_images + fixed_noise
# Clamp the *visualized* noisy images, not the input to the model during training
fixed_noisy_vis_images = torch.clamp(fixed_noisy_vis_images, -1.0, 1.0) # Adjust clamp range if needed after normalization

In [10]:
train_losses = []
step_count = 0

print(f"Starting training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    epoch_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for batch_idx, (clean_images, _) in enumerate(progress_bar):
        clean_images = clean_images.to(DEVICE) # (N, 3, IMAGE_SIZE, IMAGE_SIZE)

        # --- Generate Noisy Input for this batch ---
        noise = torch.randn_like(clean_images) * NOISE_SIGMA
        noisy_input = clean_images + noise
        noisy_input = noisy_input.to(DEVICE)

        denoised_output = model(noisy_input)

        loss = criterion(denoised_output, clean_images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        epoch_loss += current_loss
        train_losses.append(current_loss)
        step_count += 1
        progress_bar.set_postfix(loss=f"{current_loss:.4f}")

    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Training Loss: {avg_epoch_loss:.4f}")

    # --- Visualization ---
    # Visualize e.g., after first epoch and every 5 epochs thereafter, and the last epoch
    if (epoch + 1) == 1 or (epoch + 1) % 5 == 0 or (epoch + 1) == EPOCHS:
        print(f"\n--- Visualizing after Epoch {epoch+1} ---")
        model.eval()
        with torch.no_grad():
            # Use the fixed noisy images for consistent comparison
            denoised_test_output = model(fixed_noisy_vis_images.to(DEVICE)) # Use clamped noisy for vis input

        vis_clean = fixed_vis_images.cpu()
        vis_noisy = fixed_noisy_vis_images.cpu() # Already created and potentially clamped
        # Clamp the model output for visualization
        vis_denoised = torch.clamp(denoised_test_output.cpu(), -1.0, 1.0) # Adjust clamp range if needed

        num_vis = len(vis_clean) # Should be num_vis_images (e.g., 8)
        # Stack clean, noisy, denoised images for grid visualization
        # Shape becomes (3, N, C, H, W) -> permute to (N, 3, C, H, W) -> reshape
        images_to_show = torch.stack([
            vis_clean,
            vis_noisy,
            vis_denoised
        ]).permute(1, 0, 2, 3, 4).reshape(num_vis * 3, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

        visualize_images_with_titles(
            images_to_show,
            column_names=['Input', f'Noisy (σ={NOISE_SIGMA})', 'Denoised']
        )
        print("-" * 20)

# --- Final Loss Plot ---
plt.figure(figsize=(10, 5))
# Plot steps vs loss
# plt.plot(range(step_count), train_losses, alpha=0.5, label='Step Loss')
# Plot epoch average loss
epoch_ticks = [i * len(train_dataloader) for i in range(EPOCHS)]
epoch_avg_losses = [sum(train_losses[i*len(train_dataloader):(i+1)*len(train_dataloader)]) / len(train_dataloader) for i in range(EPOCHS)]
plt.plot(epoch_ticks, epoch_avg_losses, marker='o', linestyle='-', label='Avg Epoch Loss')

plt.xlabel("Training Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss Curve (Single-Step Denoising on ImageNette)")
plt.yscale('log') # Often helpful to see loss changes
plt.legend()
plt.grid(True)
plt.show()

print("Training finished.")



Starting training for 50 epochs...


Epoch 1/50:   0%|          | 0/148 [00:00<?, ?it/s]

64
128
64
128


AssertionError: Expected shape (64, 128), but got torch.Size([8, 8])

In [ ]:
PATH = f"unconditional_unet_imagenette_{IMAGE_SIZE}.pth"
torch.save(model.state_dict(), PATH)
print(f"Model saved as {PATH}")

## Out-of-Distribution Testing

In [ ]:
PATH = f"unconditional_unet_imagenette_{IMAGE_SIZE}.pth"
model = UnconditionalUNet(in_channels=IN_CHANNELS, num_hiddens=NUM_HIDDENS).to(DEVICE)
model.load_state_dict(torch.load(PATH, map_location=DEVICE))
model.eval()
print("Model loaded from", PATH)


In [ ]:
test_images_vis, _ = next(iter(vis_dataloader))
test_images_vis = test_images_vis.to(DEVICE)
# Select the first image from the batch
img = test_images_vis[0:1] # Keep batch dimension (1, C, H, W)

# --- Generate Noisy and Denoised Versions ---
sigmas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0, 1.5] # Added higher sigma
noisy_images_list = []
denoised_images_list = []

model.eval()
with torch.no_grad():
    for sigma in sigmas:
        noise = sigma * torch.randn_like(img)
        noisy = img + noise
        # Note: We don't clamp the input to the model here
        denoised = model(noisy)

        # Clamp for visualization
        noisy_vis = torch.clamp(noisy.cpu(), -1.0, 1.0) # Adjust clamp range if needed
        denoised_vis = torch.clamp(denoised.cpu(), -1.0, 1.0) # Adjust clamp range if needed

        noisy_images_list.append(noisy_vis)
        denoised_images_list.append(denoised_vis)

# --- Visualize ---
# Stack noisy and denoised images for the grid
num_sigmas = len(sigmas)
# Shape: (2, num_sigmas, C, H, W) -> permute -> (num_sigmas, 2, C, H, W) -> reshape
images_to_show = torch.stack([
    torch.cat(noisy_images_list, dim=0), # (num_sigmas, C, H, W)
    torch.cat(denoised_images_list, dim=0) # (num_sigmas, C, H, W)
]).permute(1, 0, 2, 3, 4).reshape(num_sigmas * 2, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

# Create column titles
column_names = [f"Noisy σ={s}" for s in sigmas] + [f"Denoised σ={s}" for s in sigmas]
# We'll use visualize_images_with_titles, arranging them manually might be easier here

print("Visualizing OOD Results (Different Noise Levels)")
fig, axes = plt.subplots(2, num_sigmas, figsize=(num_sigmas * 2.5, 5))

for i, sigma in enumerate(sigmas):
    # Top row: noisy image
    ax = axes[0, i]
    img_noisy_vis = denormalize(noisy_images_list[i]) # Denormalize
    img_noisy_vis = torch.clamp(img_noisy_vis, 0, 1) # Clamp to [0, 1]
    ax.imshow(img_noisy_vis.squeeze(0).permute(1, 2, 0).numpy()) # Remove batch dim, permute
    ax.axis("off")
    ax.set_title(f"Noisy σ={sigma}", fontsize=10)
    if i == 0:
        ax.set_ylabel("Input", fontsize=10)


    # Bottom row: denoised image
    ax = axes[1, i]
    img_denoised_vis = denormalize(denoised_images_list[i]) # Denormalize
    img_denoised_vis = torch.clamp(img_denoised_vis, 0, 1) # Clamp to [0, 1]
    ax.imshow(img_denoised_vis.squeeze(0).permute(1, 2, 0).numpy()) # Remove batch dim, permute
    ax.axis("off")
    ax.set_title(f"Denoised", fontsize=10)
    if i == 0:
        ax.set_ylabel("Output", fontsize=10)


plt.tight_layout()
plt.show()


# Part 2: Training a Diffusion Model

## Implementing a Time-conditioned UNet

In [ ]:
class FCBlock(nn.Module):
    def __init__(self, in_channels: int = IN_CHANNELS, out_channels: int = NUM_CLASSES):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # Definitions for layer
        self.fc = nn.Linear(in_channels, out_channels)
        self.gelu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc(x)
        x = self.gelu(x)
        return x



class TimeConditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        num_hiddens: int,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.num_hiddens = num_hiddens

        # Time embedding
        time_embed_dim = num_hiddens * 4
        self.time_embed = nn.Sequential(
            FCBlock(1, time_embed_dim),
            FCBlock(time_embed_dim, time_embed_dim)
        )

        # Projection layers for time embedding
        self.time_proj_1 = nn.Linear(time_embed_dim, num_hiddens)
        self.time_proj_2 = nn.Linear(time_embed_dim, num_hiddens)
        self.time_proj_3 = nn.Linear(time_embed_dim, num_hiddens * 2)

        # Encoder path
        self.first_conv = ConvBlock(in_channels, num_hiddens)
        self.down1 = DownBlock(num_hiddens, num_hiddens)
        self.down2 = DownBlock(num_hiddens, num_hiddens * 2)

        # Bottleneck
        self.flatten = Flatten()
        self.unflatten = Unflatten(num_hiddens * 2)

        # Decoder path
        self.upsample1 = UpBlock(num_hiddens * 4, num_hiddens)
        self.upsample2 = UpBlock(num_hiddens * 2, num_hiddens)

        # Final layers
        self.final_conv1 = ConvBlock(num_hiddens * 2, num_hiddens)
        self.final_conv2 = nn.Conv2d(num_hiddens, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor
            t: (N,) normalized time tensor
        """
        assert x.shape[-2:] == (IMAGE_SIZE, IMAGE_SIZE), \
            f"Expected input shape ({IMAGE_SIZE}, {IMAGE_SIZE}), got {x.shape[-2:]}"
        
        # Time embedding
        t = t.unsqueeze(-1).float()  # (N, 1)
        t_emb = self.time_embed(t)   # (N, time_embed_dim)

        # Project time embeddings for different layers
        t1 = self.time_proj_1(t_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens, 1, 1)
        t2 = self.time_proj_2(t_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens, 1, 1)
        t3 = self.time_proj_3(t_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens * 2, 1, 1)

        # Encoder
        x1 = self.first_conv(x)
        x1 = x1 + t1.expand_as(x1)

        x2 = self.down1(x1)
        x2 = x2 + t2.expand_as(x2)

        x3 = self.down2(x2)
        x3 = x3 + t3.expand_as(x3)

        # Bottleneck
        x_flat = self.flatten(x3)
        x_bottle = self.unflatten(x_flat)

        # Decoder with skip connections
        x = torch.cat([x_bottle, x3], dim=1)
        x = self.upsample1(x)

        x = torch.cat([x, x2], dim=1)
        x = self.upsample2(x)

        x = torch.cat([x, x1], dim=1)
        x = self.final_conv1(x)
        x = self.final_conv2(x)

        return x


## Implementing DDPM Forward and Inverse Process for Time-conditioned Denoising

In [ ]:
def ddpm_schedule(beta1: float, beta2: float, num_ts: int) -> dict:
    """Constants for DDPM training and sampling.

    Arguments:
        beta1: float, starting beta value.
        beta2: float, ending beta value.
        num_ts: int, number of timesteps.

    Returns:
        dict with keys:
            betas: linear schedule of betas from beta1 to beta2.
            alphas: 1 - betas.
            alpha_bars: cumulative product of alphas.
    """
    assert beta1 < beta2 < 1.0, "Expect beta1 < beta2 < 1.0."
    
    # Linear schedule of beta values
    betas = torch.linspace(beta1, beta2, num_ts)
    
    # Calculate alphas
    alphas = 1 - betas
    
    # Calculate cumulative product of alphas
    alpha_bars = torch.cumprod(alphas, dim=0)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alpha_bars': alpha_bars
    }

In [ ]:
def ddpm_forward(
    unet: TimeConditionalUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper.

    Args:
        unet: TimeConditionalUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        num_ts: int, number of timesteps.
    Returns:
        (,) diffusion loss.
    """
    unet.train()
    
    # Sample timestep uniformly
    t = torch.randint(0, num_ts, (x_0.shape[0],), device=x_0.device)
    
    # Get noise
    epsilon = torch.randn_like(x_0)
    
    # Get alpha_bar for the sampled timestep
    alpha_bar = ddpm_schedule['alpha_bars'][t].view(-1, 1, 1, 1)
    
    # Create noisy sample
    x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * epsilon
    
    # Predict noise
    epsilon_theta = unet(x_t, t / num_ts)
    
    # Calculate loss
    loss = F.mse_loss(epsilon_theta, epsilon)
    
    return loss

In [ ]:
@torch.inference_mode()
def ddpm_sample(
    unet: TimeConditionalUNet,
    ddpm_schedule: dict,
    img_wh: tuple[int, int],
    num_ts: int,
    seed: int = 42,
    batch_size: int = 1,
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper with classifier-free guidance.

    Args:
        unet: TimeConditionalUNet
        ddpm_schedule: dict
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.
        seed: int, random seed.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    torch.manual_seed(seed)
    DEVICE = next(unet.parameters()).device

    # Start from random noise
    x_t = torch.randn(batch_size, unet.in_channels, *img_wh, device=DEVICE)


    progress_bar = tqdm(reversed(range(num_ts)), desc="Sampling", total=num_ts, leave=False)
    for t in progress_bar:
        t_tensor = torch.tensor([t] * batch_size, device=DEVICE) 
        t_normalized = t_tensor / num_ts 
        epsilon_theta = unet(x_t, t_normalized) 

        alpha_t = ddpm_schedule['alphas'][t]
        alpha_bar_t = ddpm_schedule['alpha_bars'][t]
        beta_t = ddpm_schedule['betas'][t]

        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_alpha_bar_t = torch.sqrt(1.0 - alpha_bar_t)
        beta_over_sqrt_one_minus_alpha_bar = beta_t / sqrt_one_minus_alpha_bar_t

        if t > 0:
            noise = torch.randn_like(x_t)
            sqrt_beta_t = torch.sqrt(beta_t)
        else:
            noise = torch.zeros_like(x_t) 
            sqrt_beta_t = 0

        x_t = (1 / sqrt_alpha_t) * (
            x_t - beta_over_sqrt_one_minus_alpha_bar * epsilon_theta
        ) + sqrt_beta_t * noise

    return x_t

In [ ]:
class DDPM(nn.Module):
    def __init__(
        self,
        unet: TimeConditionalUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.num_ts = num_ts
        self.p_uncond = p_uncond

        self.ddpm_schedule = nn.ParameterDict(ddpm_schedule(betas[0], betas[1], num_ts))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        seed: int = 0,
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, img_wh, self.num_ts, seed
        )

## Training the Time-conditioned UNet

- Plot the loss curve
- Sample results on the test set

In [ ]:
# Set up for training
# dataset = MNIST(root='data', download=True, transform=ToTensor(), train=True)
# dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Setup constants and model
EPOCHS = 100 # DDPMs typically require more epochs
BATCH_SIZE = 32 # Adjust based on GPU memory and IMAGE_SIZE
LEARNING_RATE = 1e-4
NUM_HIDDENS = 64 # Increase capacity for ImageNette
NUM_TIMESTEPS = 1000 # Standard DDPM setting
BETA1, BETA2 = 1e-4, 0.02
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Initialize model and optimizer
model = TimeConditionalUNet(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    num_hiddens=NUM_HIDDENS
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

ddpm_sched = ddpm_schedule(BETA1, BETA2, NUM_TIMESTEPS)
for k, v in ddpm_sched.items():
    ddpm_sched[k] = v.to(DEVICE)

train_dataset = ImageFolder(root=os.path.join(DATASET_PATH, 'train'), transform=train_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)



In [ ]:
# Training loop
def train_epoch(model, loader, optimizer, ddpm_sched, num_ts, device=DEVICE):
    model.train()
    total_loss = 0
    progress_bar = tqdm(loader, desc='Training', leave=False)

    for batch_idx, (images, _) in enumerate(progress_bar):
        images = images.to(device)

        optimizer.zero_grad()

        # Forward pass
        loss = ddpm_forward(model, ddpm_sched, images, num_ts)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Update progress bar
        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress_bar.set_postfix({'Loss': f'{avg_loss:.4f}'})

    return total_loss / len(loader)

# Lists to store metrics
train_losses = []

# Training loop with visualization
print(f"Training TimeConditionalUNet on {DEVICE}...")

for epoch in range(EPOCHS):
    avg_loss = train_epoch(model, train_loader, optimizer, ddpm_sched, NUM_TIMESTEPS, DEVICE)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")

    if (epoch + 1) % 10 == 0 or (epoch + 1) == EPOCHS:
        print(f"\nGenerating samples after epoch {epoch+1}")
        model.eval()
        num_samples_to_show = 16
        with torch.no_grad():
            samples = ddpm_sample(
                model,
                ddpm_sched,
                img_wh=(IMAGE_SIZE, IMAGE_SIZE),
                num_ts=NUM_TIMESTEPS,
                batch_size=num_samples_to_show,
                seed=epoch
            )

            print("Generated Samples:")
            visualize_images_with_titles(samples, [f'Sample {i+1}' for i in range(num_samples_to_show)])

# Plot final loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Time-Conditional UNet Training Loss (ImageNette)')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
PATH = f"time_conditional_unet_imagenette_{IMAGE_SIZE}.pth"
torch.save(model.state_dict(), PATH)
print(f"Model saved as {PATH}")

In [ ]:
# Generate final test samples
print("\nGenerating final samples")
model.eval()
with torch.no_grad():
    # Generate multiple samples
    samples = []
    for i in range(5):  # Generate 5 batches of samples
        sample = ddpm_sample(
            model,
            ddpm_sched,
            img_wh=(IMAGE_SIZE, IMAGE_SIZE),
            num_ts=NUM_TIMESTEPS,
            seed=i
        )
        samples.append(sample)
    samples = torch.cat(samples, dim=0)
    
    # Create a grid of samples
    plt.figure(figsize=(15, 15))
    for i in range(min(25, samples.shape[0])):
        plt.subplot(5, 5, i+1)
        plt.imshow(samples[i, 0].cpu(), cmap='gray')
        plt.axis('off')
    plt.suptitle('Generated Samples')
    plt.tight_layout()
    plt.show()

# Save the model
torch.save(model.state_dict(), 'time_conditional_unet.pth')
print("Model saved as time_conditional_unet.pth")


In [ ]:
import random
# Load the saved model
model = TimeConditionalUNet(in_channels=1, num_classes=10, num_hiddens=NUM_HIDDENS).to(DEVICE)
model.load_state_dict(torch.load('time_conditional_unet.pth'))

# Generate new samples
with torch.no_grad():
    samples = ddpm_sample(
        model,
        ddpm_sched,
        img_wh=(IMAGE_SIZE, IMAGE_SIZE),
        num_ts=NUM_TIMESTEPS,
        seed=random.randint(0, 1000)  # Random seed for reproducibility
    )
    
    # Visualize
    plt.figure(figsize=(10, 10))
    for i in range(min(16, samples.shape[0])):
        plt.subplot(4, 4, i+1)
        plt.imshow(samples[i, 0].cpu(), cmap='gray')
        plt.axis('off')
    plt.tight_layout()
    plt.show()


### Implementing class-conditioned UNet

In [ ]:
class ClassConditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int = IN_CHANNELS,
        num_classes: int = NUM_CLASSES,
        num_hiddens: int = 64,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.num_hiddens = num_hiddens

        # Time embedding
        time_embed_dim = num_hiddens * 4
        self.time_embed = nn.Sequential(
            FCBlock(1, time_embed_dim),
            FCBlock(time_embed_dim, time_embed_dim)
        )

        # Class embedding
        self.class_embed = nn.Embedding(self.num_classes, time_embed_dim)

        # Combined embedding projection layers
        self.combined_proj_1 = nn.Linear(time_embed_dim * 2, num_hiddens)
        self.combined_proj_2 = nn.Linear(time_embed_dim * 2, num_hiddens)
        self.combined_proj_3 = nn.Linear(time_embed_dim * 2, num_hiddens * 2)

        # Encoder path
        self.first_conv = ConvBlock(in_channels, num_hiddens)
        self.down1 = DownBlock(num_hiddens, num_hiddens)
        self.down2 = DownBlock(num_hiddens, num_hiddens * 2)

        # Bottleneck
        self.flatten = Flatten()
        self.unflatten = Unflatten(num_hiddens * 2)

        # Decoder path
        self.upsample1 = UpBlock(num_hiddens * 4, num_hiddens)
        self.upsample2 = UpBlock(num_hiddens * 2, num_hiddens)

        # Final layers
        self.final_conv1 = ConvBlock(num_hiddens * 2, num_hiddens)
        self.final_conv2 = nn.Conv2d(num_hiddens, self.in_channels, kernel_size=3, padding=1)

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
        t: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.
            t: (N,) normalized time tensor.
            mask: (N,) mask tensor. If not None, mask out condition when mask == 0.
        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (IMAGE_SIZE, IMAGE_SIZE), \
             f"Expect input shape ({IMAGE_SIZE}, {IMAGE_SIZE}). Got {x.shape[-2:]}"
        
        # Time embedding
        t = t.unsqueeze(-1).float()  # (N, 1)
        t_emb = self.time_embed(t)   # (N, time_embed_dim)

        # Class embedding
        c_emb = self.class_embed(c)  # (N, time_embed_dim)

        # Apply masking if provided
        if mask is not None:
            mask = mask.unsqueeze(-1)  # (N, 1)
            c_emb = c_emb * mask  # Zero out embeddings for masked indices

        # Combine time and class embeddings
        combined_emb = torch.cat([t_emb, c_emb], dim=1)  # (N, time_embed_dim * 2)

        # Project combined embeddings for different layers
        emb1 = self.combined_proj_1(combined_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens, 1, 1)
        emb2 = self.combined_proj_2(combined_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens, 1, 1)
        emb3 = self.combined_proj_3(combined_emb).unsqueeze(-1).unsqueeze(-1)  # (N, num_hiddens * 2, 1, 1)

        # Encoder
        x1 = self.first_conv(x)
        x1 = x1 + emb1 

        x2 = self.down1(x1)
        x2 = x2 + emb2 

        x3 = self.down2(x2)
        x3 = x3 + emb3 

        x_flat = self.flatten(x3)
        x_bottle = self.unflatten(x_flat)

        dec1_in = torch.cat([x_bottle, x3], dim=1) 
        dec1_out = self.upsample1(dec1_in) 

        dec2_in = torch.cat([dec1_out, x2], dim=1) 
        dec2_out = self.upsample2(dec2_in) 

        dec3_in = torch.cat([dec2_out, x1], dim=1) 
        dec3_out = self.final_conv1(dec3_in) 

        output = self.final_conv2(dec3_out) 

        return output


In [ ]:
def ddpm_forward(
    unet: ClassConditionalUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    c: torch.Tensor,
    p_uncond: float,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper with classifier-free guidance."""
    unet.train()
    DEVICE = x_0.device
    N = x_0.shape[0]

    t = torch.randint(0, num_ts, (N,), device=DEVICE) 

    epsilon = torch.randn_like(x_0) 

    alpha_bar_t = ddpm_schedule['alpha_bars'][t].view(N, 1, 1, 1) 
    sqrt_alpha_bar_t = torch.sqrt(alpha_bar_t)
    sqrt_one_minus_alpha_bar_t = torch.sqrt(1.0 - alpha_bar_t)
    x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * epsilon 

    mask = torch.rand(N, device=DEVICE) > p_uncond 

    t_normalized = t / num_ts 
    epsilon_theta = unet(x_t, c, t_normalized, mask=mask) 

    loss = F.mse_loss(epsilon_theta, epsilon)

    return loss

@torch.inference_mode()
def ddpm_sample(
    unet: ClassConditionalUNet,
    ddpm_schedule: dict,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0,
    seed: int = 0,
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper with classifier-free guidance."""
    unet.eval()
    DEVICE = next(unet.parameters()).device
    N = c.shape[0]

    torch.manual_seed(seed)

    x_t = torch.randn(N, unet.in_channels, *img_wh, device=DEVICE)

    progress_bar = tqdm(reversed(range(num_ts)), desc="Sampling (CFG)", total=num_ts, leave=False)
    for t in progress_bar:
        t_tensor = torch.full((N,), t, device=DEVICE) 
        t_normalized = t_tensor / num_ts 


        mask_cond = torch.ones(N, dtype=torch.bool, device=DEVICE)
        eps_cond = unet(x_t, c, t_normalized, mask=mask_cond)


        mask_uncond = torch.zeros(N, dtype=torch.bool, device=DEVICE)
        eps_uncond = unet(x_t, c, t_normalized, mask=mask_uncond) 

        # Combine predictions using guidance scale
        epsilon_theta = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        alpha_t = ddpm_schedule['alphas'][t]
        alpha_bar_t = ddpm_schedule['alpha_bars'][t]
        beta_t = ddpm_schedule['betas'][t]

        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_alpha_bar_t = torch.sqrt(1.0 - alpha_bar_t)
        beta_over_sqrt_one_minus_alpha_bar = beta_t / sqrt_one_minus_alpha_bar_t

        if t > 0:
            noise = torch.randn_like(x_t)
            sqrt_beta_t = torch.sqrt(beta_t)
        else:
            noise = torch.zeros_like(x_t)
            sqrt_beta_t = 0

        x_t = (1 / sqrt_alpha_t) * (
            x_t - beta_over_sqrt_one_minus_alpha_bar * epsilon_theta
        ) + sqrt_beta_t * noise

    return x_t


In [ ]:
@torch.inference_mode()
def ddpm_sample(
    unet: ClassConditionalUNet,
    ddpm_schedule: dict,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0,
    seed: int = 0,
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper with classifier-free guidance."""
    unet.eval()
    
    torch.manual_seed(seed)
    
    x_t = torch.randn(c.shape[0], unet.in_channels, *img_wh, device=c.device)
    
    for t in reversed(range(num_ts)):
        t_tensor = torch.tensor([t] * c.shape[0], device=c.device)
        t_scaled = t_tensor / num_ts
        
        eps_cond = unet(x_t, c, t_scaled, mask=torch.ones_like(c, dtype=torch.bool))
        eps_uncond = unet(x_t, c, t_scaled, mask=torch.zeros_like(c, dtype=torch.bool))
        
        epsilon_theta = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        
        alpha = ddpm_schedule['alphas'][t]
        alpha_bar = ddpm_schedule['alpha_bars'][t]
        beta = ddpm_schedule['betas'][t]
        
        if t > 0:
            noise = torch.randn_like(x_t)
        else:
            noise = 0
            
        # Denoising step
        x_t = (1 / torch.sqrt(alpha)) * (
            x_t - (beta / torch.sqrt(1 - alpha_bar)) * epsilon_theta
        ) + torch.sqrt(beta) * noise
    
    return x_t

In [ ]:
class DDPM(nn.Module):
    def __init__(
        self,
        unet: ClassConditionalUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = nn.ParameterDict(ddpm_schedule(betas[0], betas[1], num_ts))

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0,
        seed: int = 0,
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, c, img_wh, self.num_ts, guidance_scale, seed
        )

## Training the Class-conditioned UNet

- Plot the loss curve
- Sample results on the test set

In [ ]:

EPOCHS = 150
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_HIDDENS = 64
NUM_TIMESTEPS = 1000
BETA1, BETA2 = 1e-4, 0.02
P_UNCOND = 0.1 
GUIDANCE_SCALE = 5.0 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

model = ClassConditionalUNet(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    num_hiddens=NUM_HIDDENS
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

ddpm_sched = ddpm_schedule(BETA1, BETA2, NUM_TIMESTEPS)
for k, v in ddpm_sched.items():
    ddpm_sched[k] = v.to(DEVICE)

train_dataset = ImageFolder(root=os.path.join(DATASET_PATH, 'train'), transform=train_transforms)
test_dataset = ImageFolder(root=os.path.join(DATASET_PATH, 'val'), transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

class_names = train_dataset.classes
print("ImageNette Classes:", class_names)


In [ ]:
def train_epoch_cfg(model, loader, optimizer, ddpm_sched, num_ts, p_uncond, device):
    model.train()
    total_loss = 0
    progress_bar = tqdm(loader, desc='Training (CFG)', leave=False)

    for batch_idx, (images, labels) in enumerate(progress_bar): # Get images and labels
        images = images.to(device)
        labels = labels.to(device) # Send labels to device

        optimizer.zero_grad()

        # Forward pass using the DDPM forward process function with CFG
        loss = ddpm_forward(model, ddpm_sched, images, labels, p_uncond, num_ts) # Pass labels and p_uncond

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / (batch_idx + 1)
        progress_bar.set_postfix({'Loss': f'{avg_loss:.4f}'})

    return total_loss / len(loader)


In [ ]:
train_losses = []
print(f"Training ClassConditionalUNet on {DEVICE}...")

for epoch in range(EPOCHS):
    # Train for one epoch
    avg_loss = train_epoch_cfg(model, train_loader, optimizer, ddpm_sched, NUM_TIMESTEPS, P_UNCOND, DEVICE)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")


    if (epoch + 1) % 10 == 0 or (epoch + 1) == EPOCHS:
        print(f"\nGenerating samples after epoch {epoch+1}")
        model.eval()
        num_classes_to_show = NUM_CLASSES 
        samples_per_class = 4 
        all_samples = []
        all_labels = []

        with torch.no_grad():
            for class_idx in range(num_classes_to_show):
                c = torch.full((samples_per_class,), class_idx, dtype=torch.long, device=DEVICE)

                samples = ddpm_sample(
                    model,
                    ddpm_sched,
                    c, 
                    img_wh=(IMAGE_SIZE, IMAGE_SIZE),
                    num_ts=NUM_TIMESTEPS,
                    guidance_scale=GUIDANCE_SCALE, 
                    seed=epoch * 100 + class_idx 
                )
                all_samples.append(samples)
                all_labels.extend([class_names[class_idx]] * samples_per_class) 

            all_samples = torch.cat(all_samples, dim=0) 

            print(f"Generated Samples (Guidance Scale: {GUIDANCE_SCALE}):")
            column_titles = [f"{class_names[i // samples_per_class]} ({i % samples_per_class + 1})" for i in range(all_samples.shape[0])]


            fig, axes = plt.subplots(num_classes_to_show, samples_per_class, figsize=(samples_per_class * 2.5, num_classes_to_show * 2.5))
            for i in range(all_samples.shape[0]):
                row = i // samples_per_class
                col = i % samples_per_class
                ax = axes[row, col]
                img_vis = denormalize(all_samples[i:i+1]) 
                img_vis = torch.clamp(img_vis, 0, 1)
                ax.imshow(img_vis.squeeze(0).permute(1, 2, 0).cpu().numpy())
                ax.axis('off')
                if col == 0:
                    ax.set_title(class_names[row], fontsize=10, loc='left', y=0.5) 

            plt.suptitle(f"Epoch {epoch+1} Samples (Guidance: {GUIDANCE_SCALE})", y=0.98)
            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.show()



plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Class-Conditional UNet Training Loss (ImageNette)')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
PATH = f"class_conditional_unet_imagenette_{IMAGE_SIZE}.pth"
torch.save(model.state_dict(), PATH)
print(f"Model saved as {PATH}")

In [ ]:
# Load the saved class-conditional UNet model
model = ClassConditionalUNet(
    in_channels=1,
    num_classes=10,
    num_hiddens=NUM_HIDDENS
).to(DEVICE)
model.load_state_dict(torch.load('class_conditional_unet.pth', map_location=DEVICE))
model.eval()
print("Model loaded from class_conditional_unet.pth")

In [ ]:
# Generate final test samples
print("\nGenerating final samples")
model.eval()
with torch.no_grad():
    # Generate multiple samples for each digit
    samples_per_digit = 5
    all_samples = []

    for digit in range(10):
        # Create condition tensor for this digit
        c = torch.full((samples_per_digit,), digit, device=DEVICE)

        # Generate samples with different guidance scales
        for scale in [1.0, 3.0, 5.0]:
            samples = ddpm_sample(
                model,
                ddpm_sched,
                c,
                img_wh=(28, 28),
                num_ts=NUM_TIMESTEPS,
                guidance_scale=scale,
                seed=digit
            )
            all_samples.append(samples)

    # Concatenate all samples
    all_samples = torch.cat(all_samples, dim=0)

    # Create a grid of samples
    plt.figure(figsize=(15, 10))
    for i in range(len(all_samples)):
        plt.subplot(10, 15, i+1)
        plt.imshow(all_samples[i, 0].cpu(), cmap='gray')
        plt.axis('off')
        if i % 15 == 0:
            plt.title(f'Digit {i//15}', y=1.0)
    plt.suptitle('Generated Samples (Columns: Different Guidance Scales)', y=0.95)
    plt.tight_layout()
    plt.show()


In [ ]:
# Function to generate samples for a specific digit
def generate_digit_samples(model, digit, num_samples=4, guidance_scale=3.0):
    model.eval()
    with torch.no_grad():
        c = torch.full((num_samples,), digit, device=DEVICE)
        samples = ddpm_sample(
            model,
            ddpm_sched,
            c,
            img_wh=(28, 28),
            num_ts=NUM_TIMESTEPS,
            guidance_scale=guidance_scale
        )
        
        plt.figure(figsize=(10, 3))
        for i in range(num_samples):
            plt.subplot(1, num_samples, i+1)
            plt.imshow(samples[i, 0].cpu(), cmap='gray')
            plt.axis('off')
        plt.suptitle(f'Generated Samples for Digit {digit}')
        plt.show()

# Example usage:
print("\nGenerating samples for specific digits:")
for digit in [0, 1, 4, 7, 9]:
    generate_digit_samples(model, digit)
